# Prediction Result Visualization

This notebook visualizes the prediction CSV produced by the model. It reads true/prediction column pairs from:

```python
CSV_PATH = "./draw\extreme_lstm_memo_reservoir_stor_4001_sof24_PL96_DM64.csv"
```

The CSV is large, so most cells load only selected variables instead of reading all 288 columns at once.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

CSV_PATH = Path(r"./draw\extreme_lstm_memo_reservoir_stor_4001_sof24_PL96_DM64.csv")
PRED_LEN = 96
NUM_VARS = 144

assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH.resolve()}"
CSV_PATH.resolve()


## Inspect File Structure

In [ ]:
header = pd.read_csv(CSV_PATH, nrows=0)
columns = header.columns.tolist()

true_cols = [c for c in columns if c.startswith("true_var_")]
pred_cols = [c for c in columns if c.startswith("pred_var_")]
var_ids = sorted(int(c.split("_")[-1]) for c in true_cols)

print(f"Number of columns: {len(columns)}")
print(f"Number of variables: {len(var_ids)}")
print(f"First columns: {columns[:8]}")
print(f"Last columns: {columns[-8:]}")
print(f"CSV path: {CSV_PATH.resolve()}")


In [ ]:
# Count rows without loading the whole file.
# This can take a few seconds for a large CSV.
with CSV_PATH.open("r", encoding="utf-8") as f:
    n_rows = sum(1 for _ in f) - 1

n_windows = n_rows // PRED_LEN
print(f"Rows: {n_rows:,}")
print(f"Pred len: {PRED_LEN}")
print(f"Approx windows if rows are flattened by forecast horizon: {n_windows:,}")


## Helper Functions

In [ ]:
def load_vars(var_ids, start_row=0, n_rows=None):
    """Load selected true/pred variable pairs from the large CSV."""
    if isinstance(var_ids, int):
        var_ids = [var_ids]
    usecols = []
    for v in var_ids:
        usecols.extend([f"true_var_{v}", f"pred_var_{v}"])
    return pd.read_csv(
        CSV_PATH,
        usecols=usecols,
        skiprows=range(1, start_row + 1) if start_row > 0 else None,
        nrows=n_rows,
    )


def metrics_np(y_true, y_pred, eps=1e-8):
    err = y_pred - y_true
    abs_err = np.abs(err)
    return {
        "MAE": float(abs_err.mean()),
        "MSE": float((err ** 2).mean()),
        "RMSE": float(np.sqrt((err ** 2).mean())),
        "MAPE": float((abs_err / (np.abs(y_true) + eps)).mean()),
        "NMAE": float(abs_err.sum() / (np.abs(y_true).sum() + eps)),
        "NRMSE": float(np.sqrt((err ** 2).sum()) / (np.sqrt((y_true ** 2).sum()) + eps)),
    }


def plot_var(var_id=0, start_row=0, n_points=1000):
    df = load_vars(var_id, start_row=start_row, n_rows=n_points)
    true = df[f"true_var_{var_id}"].to_numpy()
    pred = df[f"pred_var_{var_id}"].to_numpy()
    x = np.arange(start_row, start_row + len(df))
    m = metrics_np(true, pred)

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
    axes[0].plot(x, true, label="True", linewidth=1.5)
    axes[0].plot(x, pred, label="Pred", linewidth=1.2, alpha=0.85)
    axes[0].set_title(
        f"Variable {var_id} | MAE={m['MAE']:.6g}, RMSE={m['RMSE']:.6g}, NMAE={m['NMAE']:.4f}"
    )
    axes[0].set_ylabel("Value")
    axes[0].legend()

    axes[1].plot(x, pred - true, color="tab:red", linewidth=1)
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_ylabel("Pred - True")
    axes[1].set_xlabel("Flattened row index")

    plt.tight_layout()
    plt.show()
    return pd.Series(m)


def plot_window(var_id=0, window_id=0):
    start = int(window_id) * PRED_LEN
    return plot_var(var_id=var_id, start_row=start, n_points=PRED_LEN)


## Plot One Variable

Change `VAR_ID`, `START_ROW`, and `N_POINTS` to inspect different parts of the prediction file.


In [ ]:
VAR_ID = 143
START_ROW = 0
N_POINTS = 1200

plot_var(VAR_ID, START_ROW, N_POINTS)


## Plot One Forecast Window

If the CSV rows are flattened as `[window, horizon]`, this plots one `pred_len=96` forecast window.


In [ ]:
VAR_ID = 143
WINDOW_ID = 0

plot_window(VAR_ID, WINDOW_ID)


## Compare Multiple Variables

In [ ]:
VAR_IDS = [0, 23, 31, 107, 143]
START_ROW = 0
N_POINTS = 1000

df = load_vars(VAR_IDS, start_row=START_ROW, n_rows=N_POINTS)
x = np.arange(START_ROW, START_ROW + len(df))

fig, axes = plt.subplots(len(VAR_IDS), 1, figsize=(14, 2.4 * len(VAR_IDS)), sharex=True)
if len(VAR_IDS) == 1:
    axes = [axes]

for ax, v in zip(axes, VAR_IDS):
    true = df[f"true_var_{v}"]
    pred = df[f"pred_var_{v}"]
    mae = np.mean(np.abs(pred - true))
    ax.plot(x, true, label="True", linewidth=1.3)
    ax.plot(x, pred, label="Pred", linewidth=1.1, alpha=0.85)
    ax.set_title(f"Variable {v} | MAE={mae:.6g}")
    ax.set_ylabel("Value")

axes[0].legend(loc="upper right")
axes[-1].set_xlabel("Flattened row index")
plt.tight_layout()
plt.show()


## Per-Variable Metrics

This cell streams the CSV in chunks and computes metrics for all variables without holding the full file in memory.


In [ ]:
def compute_all_var_metrics(chunksize=50_000):
    sum_abs = np.zeros(NUM_VARS, dtype=np.float64)
    sum_sq = np.zeros(NUM_VARS, dtype=np.float64)
    sum_abs_true = np.zeros(NUM_VARS, dtype=np.float64)
    sum_sq_true = np.zeros(NUM_VARS, dtype=np.float64)
    count = 0

    true_names = [f"true_var_{i}" for i in range(NUM_VARS)]
    pred_names = [f"pred_var_{i}" for i in range(NUM_VARS)]

    for chunk in pd.read_csv(CSV_PATH, chunksize=chunksize):
        true = chunk[true_names].to_numpy(dtype=np.float64)
        pred = chunk[pred_names].to_numpy(dtype=np.float64)
        err = pred - true
        sum_abs += np.abs(err).sum(axis=0)
        sum_sq += (err ** 2).sum(axis=0)
        sum_abs_true += np.abs(true).sum(axis=0)
        sum_sq_true += (true ** 2).sum(axis=0)
        count += len(chunk)

    out = pd.DataFrame({
        "var_id": np.arange(NUM_VARS),
        "MAE": sum_abs / count,
        "MSE": sum_sq / count,
        "RMSE": np.sqrt(sum_sq / count),
        "NMAE": sum_abs / (sum_abs_true + 1e-8),
        "NRMSE": np.sqrt(sum_sq) / (np.sqrt(sum_sq_true) + 1e-8),
        "true_abs_mean": sum_abs_true / count,
    })
    return out

metrics_df = compute_all_var_metrics()
metrics_df.sort_values("MAE", ascending=False).head(15)


In [ ]:
display(metrics_df.describe())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(metrics_df["MAE"], bins=30, color="tab:blue", alpha=0.85)
axes[0].set_title("Per-variable MAE")
axes[0].set_xlabel("MAE")
axes[0].set_ylabel("Count")

axes[1].hist(metrics_df["RMSE"], bins=30, color="tab:orange", alpha=0.85)
axes[1].set_title("Per-variable RMSE")
axes[1].set_xlabel("RMSE")

axes[2].scatter(metrics_df["true_abs_mean"], metrics_df["MAE"], s=24, alpha=0.8)
axes[2].set_title("Scale vs Error")
axes[2].set_xlabel("Mean |true|")
axes[2].set_ylabel("MAE")

plt.tight_layout()
plt.show()


## Heatmaps Across 12 x 12 OD Variables

In [ ]:
mae_matrix = metrics_df.set_index("var_id")["MAE"].to_numpy().reshape(12, 12)
true_scale_matrix = metrics_df.set_index("var_id")["true_abs_mean"].to_numpy().reshape(12, 12)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(mae_matrix, cmap="viridis")
axes[0].set_title("MAE by OD pair")
axes[0].set_xlabel("Destination node")
axes[0].set_ylabel("Source node")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(true_scale_matrix, cmap="magma")
axes[1].set_title("Mean |true| by OD pair")
axes[1].set_xlabel("Destination node")
axes[1].set_ylabel("Source node")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

for ax in axes:
    ax.set_xticks(range(12))
    ax.set_yticks(range(12))

plt.tight_layout()
plt.show()


## Scatter: Prediction vs True

In [ ]:
VAR_ID = 143
N_POINTS = 50_000

df = load_vars(VAR_ID, start_row=0, n_rows=N_POINTS)
true = df[f"true_var_{VAR_ID}"].to_numpy()
pred = df[f"pred_var_{VAR_ID}"].to_numpy()

plt.figure(figsize=(6, 6))
plt.scatter(true, pred, s=4, alpha=0.25)
lo = min(true.min(), pred.min())
hi = max(true.max(), pred.max())
plt.plot([lo, hi], [lo, hi], color="black", linewidth=1)
plt.title(f"Prediction vs True | Variable {VAR_ID}")
plt.xlabel("True")
plt.ylabel("Pred")
plt.tight_layout()
plt.show()
